## choose model

In [4]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
model

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

In [56]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [6]:
len(v1)

384

In [55]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [9]:
v1.dot(dv)

np.float32(0.32332402)

In [10]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)
v2.dot(dv)

np.float32(0.019730477)

## Ingest vs encode data:

In [23]:
import os
import sys
sys.path.append("../Week_01_Agentic_RAG")
from ingest import load_faq_data
documents = load_faq_data()
print(len(documents))
print()
print(documents[0])

1350

{'id': '9e508f2212', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: When does the course start?', 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}


In [22]:
texts = []
for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

texts[0]

"Course: When does the course start? A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."

In [24]:
from tqdm.auto import tqdm
batch_size = 50 
vectors = []
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [28]:
import numpy as np 
X = np.array(vectors)
print(X)
print()
print(X.shape)

[[-0.02670621 -0.12245753  0.01594419 ... -0.00230647 -0.11218392
  -0.02365552]
 [-0.01099552 -0.11074747 -0.02536935 ...  0.09022222 -0.02697364
   0.01975675]
 [-0.08896551 -0.0612818   0.00775606 ...  0.04059711  0.00479281
  -0.02745942]
 ...
 [-0.03652928  0.01415434 -0.06838642 ...  0.04316788  0.08105534
  -0.02148627]
 [-0.13091591 -0.06990597 -0.00931883 ... -0.00044339 -0.01285729
   0.01426925]
 [-0.07984776  0.01926992  0.02544979 ... -0.03368024 -0.01884019
   0.05837058]]

(1350, 384)


## Vector search:

In [31]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)
scores = X.dot(v_query)
scores.shape

(1350,)

In [34]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.76294094))

In [35]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [37]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [38]:
scores[top5]

array([0.76294094, 0.75793695, 0.7192131 , 0.6536312 , 0.56009996],
      dtype=float32)

In [39]:
for idx in top5:
    print('Score:',scores[idx])
    print('Answer:',documents[idx])
    print()

Score: 0.76294094
Answer: {'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

Score: 0.75793695
Answer: {'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

Score: 0.7192131
Answer: {'id': '41aabbd7c5', 'course': 'machine-learnin

## Minsearch Vector Search

In [41]:
from minsearch import VectorSearch
vindex = VectorSearch(keyword_fields = ['course'])
vindex.fit(X, documents)                                  

In [42]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [44]:
results = vindex.search(query_vector, filter_dict = {"course":"llm-zoomcamp"},num_results = 5)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

## RAG with vector search:

In [45]:
from dotenv import load_dotenv
import os 
load_dotenv()

True

In [46]:
from ingest import load_faq_data, build_index
index = build_index(documents)
index

In [48]:
from rag_helper import RAGBase
from google import genai

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
assistant = RAGBase(index = index, llm_client = client)
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [49]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [52]:
vector_assistant = RAGVector(embedder = model, index = vindex, llm_client = client)
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join the course even if it has already begun.\n\nHowever, if you wish to receive a certificate, you need to submit your project while submissions are still being accepted. You can also start learning and submitting homework without formally registering, as registration is primarily to gauge interest before the start date.'

## Vector search sqlite seardch

In [68]:
from sqlitesearch import VectorSearchIndex
sql_index = VectorSearchIndex(keyword_fields = ['course'], mode = 'ivf', db_path = 'faq_vectors2.db')
sql_index.fit(vectors, documents)

In [69]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = sql_index.search(query_vector, num_results=5)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [70]:
results = sql_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

In [71]:
sql_index.close()

## Reopen if close:

In [74]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db")
vs_index

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [76]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)
results

[{'id': '5ca6890c1a',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: How to run producer/consumer/kstreams/etc in terminal',
  'answer': 'In the project directory, run:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```'},
 {'id': 'cd8a62fc55',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent',
  'answer': 'For example, when running `JsonConsumer.java`, you might see:\n\n```\nConsuming form kafka started\n\nRESULTS:::0\n\nRESULTS:::0\n\nRESULTS:::0\n```\n\nOr when running `JsonProducer.java`, you might encounter:\n\n```\nException in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed\n```\n\n**Solution:**\n\n1. Ensure the `StreamsConfig.BO